# C — Model akhir & submission (Seno) — strategi v3

Notebook bagian C, Task 2 Datavidia 11.0.

**Alur v3:** cluster dari model (B, `fahmi/v3`) → **nama cluster diputuskan anggota tim** (`cluster_names.json`) →
seed = foto dekat pusat cluster → label spreading → Logistic Regression dari **embedding per foto saja** (tanpa `event_id`) → submission.

- Tidak ada label per foto. Nama hanya per cluster, ditulis eksplisit sebagai dictionary (bagian 5) untuk notebook final.
- Cluster `ragu` tetap dipakai dengan bobot lebih kecil.
- Nama cluster tidak diubah setelah melihat skor leaderboard.

**Output** (folder run baru, tidak menimpa): `model.joblib`, `pred_train.csv`, `submission_runN.csv`, `meta.json`.

## 0. Library & versi

In [ ]:
import json, os, platform
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow, scipy, sklearn, joblib

from sklearn.decomposition import PCA
from sklearn.semi_supervised import LabelSpreading
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

VERSI_LIB = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
             "pyarrow": pyarrow.__version__, "scipy": scipy.__version__,
             "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
for k, v in VERSI_LIB.items():
    print(f"{k:13s} {v}")

## Pengaturan
Semua path dan angka di sini. Path relatif terhadap folder `seno/`.

In [ ]:
# --- Path input ---
KANDIDAT_SAMPLE_SUB = ["../../Task-2/sample_submission.csv",   # susunan laptop Seno
                       "../../sample_submission.csv",          # susunan laptop Fahmi
                       "../Dataset/sample_submission.csv"]     # susunan README repo
FITUR_DIR = "../hafizh/v3.1"   # embedding DINOv2-L FP32 (harus sama dengan input B)
LABEL_DIR = "../fahmi/v3"      # events.parquet + cluster_names.json + centroids.npy dari B
RUN_PEMBANDING = "run2"        # run sebelumnya, hanya untuk perbandingan

# --- Output (jika sudah ada, dipakai run4, run5, ... ; tidak menimpa) ---
OUTPUT_DIR = "run3"

SEED = 42
KELAS = ["Earthquake", "Explosion", "Fire", "Hurricane", "Tsunami", "Volcano"]

# --- Seed & bobot sampel ---
SEED_KEDEKATAN_MIN = 0.5                    # foto dengan kedekatan >= 0.5 (setengah terdekat pusat) jadi seed
BOBOT_YAKIN = {"yakin": 1.0, "ragu": 0.5}   # pengali bobot per tingkat yakin cluster
# bobot sampel = kedekatan x BOBOT_YAKIN[yakin]

# --- Label spreading ---
LS_KNN = 15
LS_ALPHA = 0.2

# --- Classifier (fitur per foto saja, tanpa event_id) ---
FITUR_BLOK = ["color_pre", "color_post"]   # file dino_<blok>.npy
C_PCA = 128
LR_C = 1.0
CV_FOLD = 5

SAMPLE_SUB = next((p for p in KANDIDAT_SAMPLE_SUB if os.path.exists(p)), None)
np.random.seed(SEED)
for p in [SAMPLE_SUB, FITUR_DIR, LABEL_DIR]:
    print(f"{'ada ' if p and os.path.exists(p) else 'TIDAK ADA'}  {p}")
assert SAMPLE_SUB is not None, "sample_submission.csv tidak ketemu: tambahkan path-nya ke KANDIDAT_SAMPLE_SUB"
assert os.path.exists(LABEL_DIR), f"{LABEL_DIR} belum ada: tunggu Fahmi push fahmi/v3, lalu git pull"

## 1. Muat data & cek input

In [ ]:
sub    = pd.read_csv(SAMPLE_SUB)
events = pd.read_parquet(f"{LABEL_DIR}/events.parquet")
with open(f"{LABEL_DIR}/cluster_names.json", encoding="utf-8") as f:
    nama_raw = json.load(f)
with open(f"{LABEL_DIR}/meta.json", encoding="utf-8") as f:
    meta_b = json.load(f)

ids = sub["Id"].to_numpy()
N = len(ids)
KOLOM_TARGET = sub.columns[1]

names = pd.DataFrame.from_dict(nama_raw, orient="index")
names.index = names.index.astype(int)
names.index.name = "event_id"
names = names.sort_index()

# Urutan Id
assert sub["Id"].is_unique and KOLOM_TARGET == "Phenomenon"
assert len(events) == N and (events["Id"].to_numpy() == ids).all(), "urutan Id events.parquet beda dengan sample_submission"

# Nama cluster: lengkap, valid, sudah diputuskan
assert set(events["event_id"]) == set(names.index), \
    f"cluster tanpa nama: {sorted(set(events['event_id']) - set(names.index))} | nama tanpa cluster: {sorted(set(names.index) - set(events['event_id']))}"
tidak_valid = names.loc[~names["kelas"].isin(KELAS), "kelas"]
assert tidak_valid.empty, f"kelas tidak valid (cluster 'campur' harus dipecah & dinamai dulu): {tidak_valid.to_dict()}"
if "yakin" not in names:
    names["yakin"] = "yakin"
names["yakin"] = names["yakin"].fillna("yakin").replace("", "yakin").str.lower()
assert names["yakin"].isin(list(BOBOT_YAKIN)).all(), f"nilai yakin harus salah satu dari {list(BOBOT_YAKIN)}"
for kol in ["penilai", "alasan"]:
    if kol not in names:
        names[kol] = None
NAMA_DUMMY = bool(names["penilai"].map(lambda p: isinstance(p, list) and "DUMMY" in p).any())
if NAMA_DUMMY:
    print("PERINGATAN: nama cluster masih DUMMY (uji kode) -> file submission TIDAK akan dibuat")

# Fitur B harus sama dengan FITUR_DIR (jika dicatat di meta)
fitur_b = meta_b.get("input", {}).get("fitur")
if fitur_b is not None:
    assert fitur_b == FITUR_DIR.replace("../", ""), f"B memakai {fitur_b}, bukan {FITUR_DIR}"

# Embedding
emb = {}
for b in sorted(set(FITUR_BLOK) | {"color_pre", "color_post"}):
    emb[b] = np.load(f"{FITUR_DIR}/dino_{b}.npy")
    assert emb[b].shape == (N, 1024) and not np.isnan(emb[b]).any(), f"embedding {b} bermasalah"

event_id = events["event_id"].to_numpy()
nama_cluster = names["kelas"].reindex(event_id).to_numpy()   # nama kelas per foto dari cluster-nya
kelas_hilang = [k for k in KELAS if k not in set(names["kelas"])]
print("N =", N, "| cluster:", len(names), "| ragu:", int((names["yakin"] == "ragu").sum()))
if kelas_hilang:
    print("PERINGATAN: kelas tanpa cluster ->", kelas_hilang, "(classifier tidak akan pernah memprediksinya; macro F1 kelas itu = 0)")
print("Semua cek input: LULUS")

In [ ]:
# Ringkasan nama cluster dari tim, per kelas
ukuran = pd.Series(event_id).value_counts()
names["n"] = ukuran.reindex(names.index).astype(int)
pd.DataFrame({
    "n_cluster": names["kelas"].value_counts(),
    "n_cluster_ragu": names.loc[names["yakin"] == "ragu", "kelas"].value_counts(),
    "n_foto": names.groupby("kelas")["n"].sum(),
}).reindex(KELAS).fillna(0).astype(int)

In [ ]:
# Daftar cluster: nama, tingkat yakin, ukuran, penilai, alasan
pd.set_option("display.max_colwidth", 70)
names[["kelas", "yakin", "n", "penilai", "alasan"]]

## 2. Ruang Z, seed, dan bobot
Ruang `Z` dibangun seperti B: L2 per blok → gabung pre+post → kurangi rata-rata → PCA → bulatkan.
Cek: pusat cluster di ruang ini harus sama dengan `centroids.npy` dari B.

In [ ]:
def l2(a):
    a = a.astype(np.float64)
    return a / np.linalg.norm(a, axis=1, keepdims=True)

BP = meta_b.get("pengaturan", {})
B_PCA, B_ROUND = int(BP.get("pca", 64)), int(BP.get("round", 4))
X_b = np.hstack([l2(emb["color_pre"]), l2(emb["color_post"])])
if os.path.exists(f"{LABEL_DIR}/pca.joblib") and os.path.exists(f"{LABEL_DIR}/mean_x.npy"):
    # Pakai PCA yang disimpan B (tanpa fit ulang): hasil fit PCA bisa sedikit beda antar laptop
    pca_b = joblib.load(f"{LABEL_DIR}/pca.joblib")
    mean_x = np.load(f"{LABEL_DIR}/mean_x.npy")
    assert np.allclose(mean_x, X_b.mean(axis=0), atol=1e-6), "mean_x B beda dengan fitur di FITUR_DIR"
    Z = np.round(pca_b.transform(X_b - mean_x), B_ROUND)
    SUMBER_Z = "pca.joblib + mean_x.npy dari B"
else:
    Z = np.round(PCA(B_PCA, random_state=SEED).fit_transform(X_b - X_b.mean(axis=0)), B_ROUND)
    SUMBER_Z = "PCA fit ulang"
print(f"Z: {Z.shape} (PCA {B_PCA}, round {B_ROUND}) | sumber: {SUMBER_Z}")

# Cek ruang Z terhadap centroid B (baris = event_id terurut)
ev_urut = np.sort(names.index.to_numpy())
C_z = np.vstack([Z[event_id == e].mean(axis=0) for e in ev_urut])
CEK_CENTROID = "tidak dicek"
if os.path.exists(f"{LABEL_DIR}/centroids.npy"):
    C_b = np.load(f"{LABEL_DIR}/centroids.npy")
    if C_b.shape == C_z.shape:
        selisih = float(np.abs(C_b - C_z).max())
        CEK_CENTROID = f"selisih maks {selisih:.2e}"
        assert selisih < 1e-3, f"ruang Z beda dengan B (selisih centroid {selisih:.3g})"
        print("Centroid B tereproduksi:", CEK_CENTROID)
    else:
        CEK_CENTROID = f"bentuk beda {C_b.shape} vs {C_z.shape}"
        print("PERINGATAN: bentuk centroids.npy", C_b.shape, "beda dengan", C_z.shape, "-> cek kontrak dengan Fahmi")

# Kedekatan ke pusat (0-1, 1 = paling dekat): pakai dari B, atau hitung sendiri jika tidak ada
if "kedekatan" in events:
    kedekatan = events["kedekatan"].to_numpy(dtype=float)
else:
    kedekatan = np.zeros(N)
    for e in ev_urut:
        ix = np.where(event_id == e)[0]
        r = np.empty(len(ix)); r[np.argsort(np.linalg.norm(Z[ix] - Z[ix].mean(axis=0), axis=1), kind="stable")] = np.arange(len(ix))
        kedekatan[ix] = 1 - r / max(len(ix) - 1, 1)
    print("kedekatan dihitung dari Z (kolom tidak ada di events.parquet)")
assert np.isfinite(kedekatan).all() and kedekatan.min() >= 0 and kedekatan.max() <= 1

is_seed = kedekatan >= SEED_KEDEKATAN_MIN
bobot = kedekatan * names["yakin"].map(BOBOT_YAKIN).reindex(event_id).to_numpy(dtype=float)
print(f"Seed: {int(is_seed.sum())} foto | bobot rata-rata {bobot.mean():.3f}")
pd.Series(nama_cluster[is_seed]).value_counts().reindex(KELAS).fillna(0).astype(int).to_frame("n_seed")

## 3. Label spreading
Graf kNN di ruang `Z`. Foto seed diberi nama cluster-nya; foto lain (-1) diisi oleh label spreading.

In [ ]:
y = np.full(N, -1)
y[is_seed] = pd.Categorical(nama_cluster[is_seed], categories=KELAS).codes
ls = LabelSpreading(kernel="knn", n_neighbors=LS_KNN, alpha=LS_ALPHA, max_iter=1000).fit(Z, y)
y_latih = np.array(KELAS)[ls.classes_][ls.label_distributions_.argmax(axis=1)]
print(f"Kelas terisi: {len(set(y_latih))}/6 | label latih sama dengan nama cluster: {(y_latih == nama_cluster).mean():.3f}")
pd.DataFrame({"nama_cluster": pd.Series(nama_cluster).value_counts(),
              "label_latih": pd.Series(y_latih).value_counts()}).reindex(KELAS).fillna(0).astype(int)

## 4. Classifier
Fitur per foto = blok embedding `FITUR_BLOK` (L2 per blok, digabung). **Tidak** memakai `event_id`.
Pipeline: StandardScaler → PCA → LogisticRegression (`class_weight="balanced"`), bobot sampel = kedekatan × tingkat yakin.

CV mengukur konsistensi classifier terhadap label latih, **bukan** skor leaderboard.

In [ ]:
def bangun_fitur(emb, blok):
    return np.hstack([l2(emb[b]) for b in blok])

def buat_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(C_PCA, random_state=SEED)),
        ("clf", LogisticRegression(class_weight="balanced", C=LR_C, max_iter=5000)),
    ])

def latih(X, y, w):
    return buat_pipeline().fit(X, y, clf__sample_weight=w)

X_c = bangun_fitur(emb, FITUR_BLOK)
print("Fitur classifier:", X_c.shape)

cv = StratifiedKFold(CV_FOLD, shuffle=True, random_state=SEED)
pred_cv = np.empty(N, dtype=object)
for tr, te in cv.split(X_c, y_latih):
    pred_cv[te] = latih(X_c[tr], y_latih[tr], bobot[tr]).predict(X_c[te])
CV = {"acc": round(float(accuracy_score(y_latih, pred_cv)), 4),
      "macro_f1": round(float(f1_score(y_latih, pred_cv, average="macro")), 4)}
print(f"CV {CV_FOLD}-fold vs label latih: acc {CV['acc']:.3f} | macro-F1 {CV['macro_f1']:.3f}")

model = latih(X_c, y_latih, bobot)
pred = model.predict(X_c)
proba = model.predict_proba(X_c)
print("Urutan kelas model:", [str(k) for k in model.classes_])

In [ ]:
HASIL = {
    "kelas_terprediksi": int(len(set(pred))),
    "sama_label_latih": round(float((pred == y_latih).mean()), 4),
    "sama_nama_cluster": round(float((pred == nama_cluster).mean()), 4),
}
if os.path.exists(f"{RUN_PEMBANDING}/pred_train.csv"):
    ref = pd.read_csv(f"{RUN_PEMBANDING}/pred_train.csv")
    assert (ref["Id"].to_numpy() == ids).all()
    HASIL[f"sama_{RUN_PEMBANDING}"] = round(float((ref["Phenomenon"].to_numpy() == pred).mean()), 4)
print(HASIL)
distribusi = pd.DataFrame({
    "nama_cluster": pd.Series(nama_cluster).value_counts(),
    "label_latih": pd.Series(y_latih).value_counts(),
    "prediksi": pd.Series(pred).value_counts(),
}).reindex(KELAS).fillna(0).astype(int)
distribusi["porsi_prediksi"] = (distribusi["prediksi"] / N).round(3)
distribusi

In [ ]:
# Cluster yang prediksinya paling banyak berbeda dari nama cluster (untuk dicek, BUKAN untuk mengubah nama)
per_cluster = pd.DataFrame({"event_id": event_id, "sama": pred == nama_cluster}).groupby("event_id")["sama"].mean()
cek = names[["kelas", "yakin", "n"]].assign(porsi_prediksi_sama=per_cluster.round(3))
cek.sort_values("porsi_prediksi_sama").head(10)

## 5. Dictionary nama cluster (untuk notebook final)
Salin sel keluaran di bawah ke `final/01_model.ipynb`. Nama ditulis eksplisit per cluster; alasannya ada di write-up.

In [ ]:
print("NAMA_CLUSTER = {")
for e, r in names.iterrows():
    print(f'    {e}: "{r["kelas"]}",  # {r["yakin"]}')
print("}")

## 6. Simpan output
Folder run baru; jika `OUTPUT_DIR` sudah ada, dipakai nomor berikutnya (tidak menimpa).
File submission hanya dibuat jika nama cluster bukan DUMMY.

In [ ]:
nomor = int(OUTPUT_DIR.lstrip("run"))
OUT = f"run{nomor}"
while os.path.exists(OUT):
    nomor += 1
    OUT = f"run{nomor}"
os.makedirs(OUT)

joblib.dump({"pipeline": model, "fitur_blok": FITUR_BLOK, "l2_per_blok": True,
             "kelas": [str(k) for k in model.classes_], "versi_lib": VERSI_LIB},
            f"{OUT}/model.joblib")

pred_train = pd.DataFrame({"Id": ids, "Phenomenon": pred, "label_latih": y_latih,
                           "is_seed": is_seed, "event_id": event_id, "bobot": bobot.round(6)})
for i, k in enumerate(model.classes_):
    pred_train[f"proba_{k}"] = proba[:, i].round(6)
pred_train.to_csv(f"{OUT}/pred_train.csv", index=False)

if not NAMA_DUMMY:
    submission = sub[["Id"]].copy()
    submission[KOLOM_TARGET] = pred
    submission.to_csv(f"{OUT}/submission_{OUT}.csv", index=False)

meta = {
    "versi": OUT,
    "oleh": "Seno",
    "tanggal": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "strategi": "v3: cluster dari model, nama per cluster dari anggota tim, prediksi dari classifier",
    "input": {"fitur": FITUR_DIR.replace("../", ""), "label": LABEL_DIR.replace("../", ""),
              "nama_dummy": NAMA_DUMMY, "cek_centroid": CEK_CENTROID, "sumber_z": SUMBER_Z},
    "pengaturan": {"seed_kedekatan_min": SEED_KEDEKATAN_MIN, "bobot_yakin": BOBOT_YAKIN,
                   "b_pca": B_PCA, "b_round": B_ROUND, "ls_kernel": "knn", "ls_knn": LS_KNN, "ls_alpha": LS_ALPHA,
                   "fitur_blok": FITUR_BLOK, "c_pca": C_PCA, "lr_C": LR_C,
                   "class_weight": "balanced", "seed": SEED},
    "nama_cluster": {int(e): r["kelas"] for e, r in names.iterrows()},
    "n_seed": int(is_seed.sum()),
    "cv_vs_label_latih": CV,
    "hasil": HASIL,
    "distribusi_prediksi": {k: int(distribusi.loc[k, "prediksi"]) for k in KELAS},
    "versi_lib": VERSI_LIB,
}
with open(f"{OUT}/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Disimpan ke", OUT, ":", sorted(os.listdir(OUT)))

In [ ]:
# Cek ulang dari file: muat model -> prediksi ulang tanpa .fit() -> harus sama 100%
m = joblib.load(f"{OUT}/model.joblib")
assert (m["pipeline"].predict(bangun_fitur(emb, m["fitur_blok"])) == pred).all(), "prediksi dari model tersimpan beda"
print("Prediksi model tersimpan = prediksi training: 100%")
if NAMA_DUMMY:
    print("Nama DUMMY: submission tidak dibuat. Jalankan ulang setelah cluster_names.json final dari tim.")
else:
    cek_sub = pd.read_csv(f"{OUT}/submission_{OUT}.csv")
    assert list(cek_sub.columns) == list(sub.columns) and (cek_sub["Id"] == sub["Id"]).all()
    assert cek_sub[KOLOM_TARGET].notna().all() and set(cek_sub[KOLOM_TARGET]) <= set(KELAS)
    print("Format submission: LULUS")
    print(f"Pesan commit: [C] {OUT}: LogReg, nama cluster tim ({LABEL_DIR.replace('../', '')})")